# 3. Работа с файлом JSON в pandas

Используя библиотеку pandas, напишите скрипт, который

- загружает файл JSON с названием steam_games.json
- на его основе создает объект Dataframe
- выполняет фильтрацию игр по жанру Action, оценке Positive,  столбцу owners > 10000
- создает сводную таблицу с подсчетом количества игр на платформах (platforms)
- сортирует по количеству игр
- сохраняет полученный датафрейм в новый файл с именем steam_games_analyze.json

Источник данных - https://www.kaggle.com/datasets/tristan581/all-55000-games-on-steam-november-2022

In [1]:
import pandas as pd

In [2]:
steam_games = pd.read_json("data/task3/steam_games.json", orient="index")
# Ориентация по индексу, так как данные в JSON представлены в виде словаря, где ключи - это индексы, а значения - это строки данных.

# Чтение JSON в pandas (`pd.read_json`)

При чтении JSON-файлов или строк с помощью функции `pd.read_json()` ключевую роль играет параметр **`orient`**. Он подсказывает pandas, как именно распарсить структуру JSON и правильно собрать из неё DataFrame.

### Таблица ориентаций (`orient`)

| Значение | Что означает |
| :--- | :--- |
| **'records'** | Каждая строка DataFrame преобразуется в отдельный словарь, где ключи — названия столбцов, а значения — данные строки. Получается список таких словарей. Часто используется для передачи данных в API, где ожидается массив объектов. |
| **'index'** | Индекс DataFrame используется как ключи JSON, а значения — это словари с названиями столбцов и данными строки. |
| **'columns'** | По умолчанию. Каждая колонка DataFrame становится ключом, а значением — список элементов этого столбца. |
| **'split'** | Результат — словарь с тремя ключами: 'index' (список индексов), 'columns' (список названий столбцов) и 'data' (список списков значений ячеек). Этот формат помогает легко восстановить исходный DataFrame. |
| **'table'** | Используется для сложных структур, например, MultiIndex. Сохраняет схему и данные в виде вложенного словаря. Однако при работе с одноуровневым MultiIndex могут возникать проблемы (например, появление NaN в индексе). |
| **'values'** | Просто массив всех значений DataFrame. |

### 3 важных нюанса при чтении (`read_json`)

1. **Формат JSON Lines (NDJSON):**
   Если ваш файл представляет собой не один большой JSON-объект, а множество маленьких JSON-строк, разделенных переносом каретки (каждая строка — отдельная запись), обязательно используйте параметр **`lines=True`**:
   ```python
   # Чтение файла, где каждая строка - это отдельный JSON (обычно в формате 'records')
   df = pd.read_json('data.jsonl', lines=True, orient='records')
   ```

2. **Явное указание `orient`:**
   Хотя для `orient='columns'` pandas использует автоопределение, для форматов `'records'` и `'split'` **всегда лучше указывать `orient` явно**, иначе pandas может неправильно интерпретировать данные или выдать ошибку.

3. **Сохранение типов данных:**
   При чтении JSON pandas может привести все строковые даты к типу `object`. Чтобы сразу перевести их в формат datetime, используйте параметр `convert_dates`:
   ```python
   df = pd.read_json('data.json', orient='records', convert_dates=['date_column'])
   ```

In [3]:
steam_games.head()

,appid,name,short_description,developer,publisher,genre,tags,type,categories,owners,...,price,initialprice,discount,ccu,languages,platforms,release_date,required_age,website,header_image
10,10,Counter-Strike,Play the world's number 1 online action game. ...,Valve,Valve,Action,"{'Action': 5426, 'FPS': 4831, 'Multiplayer': 3...",game,"[Multi-player, Valve Anti-Cheat enabled, Onlin...","10,000,000 .. 20,000,000",...,999,999,0,13990,"English, French, German, Italian, Spanish - Sp...","{'windows': True, 'mac': True, 'linux': True}",2000/11/1,0,,https://cdn.akamai.steamstatic.com/steam/apps/...
1000000,1000000,ASCENXION,ASCENXION is a 2D shoot 'em up game where you ...,IndigoBlue Game Studio,PsychoFlux Entertainment,"Action, Adventure, Indie","{'Shoot 'Em Up': 186, 'Metroidvania': 181, 'Bu...",game,"[Single-player, Partial Controller Support, St...","0 .. 20,000",...,999,999,0,0,"English, Korean, Simplified Chinese","{'windows': True, 'mac': False, 'linux': False}",2021/05/14,0,,https://cdn.akamai.steamstatic.com/steam/apps/...
1000010,1000010,Crown Trick,"Enter a labyrinth that moves as you move, wher...",NEXT Studios,"Team17, NEXT Studios","Adventure, Indie, RPG, Strategy","{'Rogue-like': 268, 'Turn-Based Combat': 254, ...",game,"[Single-player, Partial Controller Support, St...","200,000 .. 500,000",...,599,1999,70,99,"Simplified Chinese, English, Japanese, Traditi...","{'windows': True, 'mac': False, 'linux': False}",2020/10/16,0,,https://cdn.akamai.steamstatic.com/steam/apps/...
1000030,1000030,"Cook, Serve, Delicious! 3?!","Cook, serve and manage your food truck as you ...",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy","{'Typing': 221, 'Management': 213, 'Casual': 2...",game,"[Multi-player, Single-player, Co-op, Steam Ach...","100,000 .. 200,000",...,1999,1999,0,76,English,"{'windows': True, 'mac': True, 'linux': False}",2020/10/14,0,http://www.cookservedelicious.com,https://cdn.akamai.steamstatic.com/steam/apps/...
1000040,1000040,细胞战争,这是一款打击感十足的细胞主题游戏！操作简单但活下去却不简单，“你”作为侵入人体的细菌病毒，通...,DoubleC Games,DoubleC Games,"Action, Casual, Indie, Simulation","{'Action': 22, 'Casual': 22, 'Indie': 21, 'Sim...",game,[Single-player],"0 .. 20,000",...,199,199,0,0,Simplified Chinese,"{'windows': True, 'mac': False, 'linux': False}",2019/03/30,0,,https://cdn.akamai.steamstatic.com/steam/apps/...


In [4]:
steam_games.columns

Index(['appid', 'name', 'short_description', 'developer', 'publisher', 'genre',
       'tags', 'type', 'categories', 'owners', 'positive', 'negative', 'price',
       'initialprice', 'discount', 'ccu', 'languages', 'platforms',
       'release_date', 'required_age', 'website', 'header_image'],
      dtype='str')

In [5]:
steam_games[["genre", "positive", "negative", "owners"]]

,genre,positive,negative,owners
10,Action,201215,5199,"10,000,000 .. 20,000,000"
1000000,"Action, Adventure, Indie",27,5,"0 .. 20,000"
1000010,"Adventure, Indie, RPG, Strategy",4032,646,"200,000 .. 500,000"
1000030,"Action, Indie, Simulation, Strategy",1575,115,"100,000 .. 200,000"
1000040,"Action, Casual, Indie, Simulation",0,1,"0 .. 20,000"
...,...,...,...,...
999880,Education,12,3,"0 .. 20,000"
999890,"Adventure, Casual, Indie",2,0,"0 .. 20,000"
999900,Animation & Modeling,12,19,"20,000 .. 50,000"
999930,"Indie, RPG, Strategy, Early Access",2,2,"0 .. 20,000"


In [6]:
steam_games[["genre", "positive", "negative", "owners"]].info()

<class 'pandas.DataFrame'>
Index: 55691 entries, 10 to 999990
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   genre     55691 non-null  str  
 1   positive  55691 non-null  int64
 2   negative  55691 non-null  int64
 3   owners    55691 non-null  str  
dtypes: int64(2), str(2)
memory usage: 4.1 MB


In [7]:
# Разделяем столбец "owners" на два отдельных столбца "owners_from" и "owners_to" с помощью регулярного выражения, где данные вида: 20,000 .. 100,000
steam_games[["owners_from", "owners_to"]] = steam_games["owners"].str.extract(r"^\s*([\d,]+)\s*\.\.\s*([\d,]+)\s*$")

In [8]:
# Преобразование строковых значений в числовые, удаляя запятые и приводя к типу int
steam_games["owners_from"] = steam_games["owners_from"].str.replace(",", "").astype(int)
steam_games["owners_to"] = steam_games["owners_to"].str.replace(",", "").astype(int)

In [9]:
filtered = steam_games[(steam_games["genre"] == "Action") & (steam_games["positive"] > steam_games["negative"]) & (steam_games["owners_from"] > 10000)]

In [10]:
filtered[["genre", "positive", "negative", "owners", "owners_from", "owners_to"]]

,genre,positive,negative,owners,owners_from,owners_to
10,Action,201215,5199,"10,000,000 .. 20,000,000",10000000,20000000
1002970,Action,14,4,"200,000 .. 500,000",200000,500000
1003480,Action,471,57,"20,000 .. 50,000",20000,50000
1007040,Action,6686,372,"500,000 .. 1,000,000",500000,1000000
10090,Action,39281,3493,"2,000,000 .. 5,000,000",2000000,5000000
...,...,...,...,...,...,...
976730,Action,181246,14148,"5,000,000 .. 10,000,000",5000000,10000000
977050,Action,81,24,"20,000 .. 50,000",20000,50000
987400,Action,283,37,"20,000 .. 50,000",20000,50000
991560,Action,573,543,"20,000 .. 50,000",20000,50000


In [11]:
filtered["platforms"]

10           {'windows': True, 'mac': True, 'linux': True}
1002970    {'windows': True, 'mac': False, 'linux': False}
1003480     {'windows': True, 'mac': True, 'linux': False}
1007040    {'windows': True, 'mac': False, 'linux': False}
10090      {'windows': True, 'mac': False, 'linux': False}
                                ...                       
976730     {'windows': True, 'mac': False, 'linux': False}
977050     {'windows': True, 'mac': False, 'linux': False}
987400     {'windows': True, 'mac': False, 'linux': False}
991560     {'windows': True, 'mac': False, 'linux': False}
99300      {'windows': True, 'mac': False, 'linux': False}
Name: platforms, Length: 712, dtype: object

In [12]:
filtered["platforms"] = filtered["platforms"].apply(
    lambda x: [platform for platform, is_true in x.items() if is_true]
)

In [13]:
filtered.info()

<class 'pandas.DataFrame'>
Index: 712 entries, 10 to 99300
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   appid              712 non-null    int64 
 1   name               712 non-null    str   
 2   short_description  712 non-null    str   
 3   developer          712 non-null    str   
 4   publisher          712 non-null    str   
 5   genre              712 non-null    str   
 6   tags               712 non-null    object
 7   type               712 non-null    str   
 8   categories         712 non-null    object
 9   owners             712 non-null    str   
 10  positive           712 non-null    int64 
 11  negative           712 non-null    int64 
 12  price              712 non-null    int64 
 13  initialprice       712 non-null    int64 
 14  discount           712 non-null    int64 
 15  ccu                712 non-null    int64 
 16  languages          712 non-null    str   
 17  platforms 

In [14]:
pivot = filtered.explode("platforms").pivot_table(index="platforms", values="name", aggfunc="count")
# .explode("platforms") - разбивает список платформ отдельные элементы, чтобы группировка была возможна

# platform_counts = filtered.explode("platforms").groupby("platforms").size()  # Аналог через группировку и подсчет количества строк в каждой группе

In [15]:
pivot.sort_values("name")

,name
platforms,
linux,74
mac,107
windows,711


In [16]:
pivot.to_json("data/task3/steam_games_analyze.json.json", orient="index")